# US equities panel: finding the few things three thousand stocks have in common

Every model so far has predicted each stock from that stock's own features. But stocks do not
move independently - most of what a broad panel does on any day is one thing happening to all of
it, and a handful of further things happening to overlapping groups of it. A **latent factor**
is one of those common movements: not a column anybody computed, but a pattern extracted from
how the returns move together, with each stock carrying a **loading** saying how much of that
pattern it takes.

Two ways of extracting them are fitted here, and the difference between them is the whole
lesson:

- [`13a_pca`](13a_pca.ipynb) takes the factors from the return panel alone. Principal component
  analysis asks which combinations of stocks account for the most common variation, and answers
  without being told anything about the stocks. A loading is then a number attached to a stock,
  fitted over the training window and carried forward.
- [`13b_ipca`](13b_ipca.ipynb) conditions the loadings on what the stocks *are*. Instrumented
  principal components makes a stock's loading a function of its observable characteristics, so
  two stocks with the same characteristics load the same way and a stock whose characteristics
  change has its loading change with them.


<!-- ML4T_ZH_BEGIN c0000-p000 -->
# 美國股票面板：找出三千檔股票共同的少數變動

此前每個模型都根據個別股票自身的特徵來預測該股票。但股票不會獨立變動：大型面板在任何一天的變化，大部分來自影響全部股票的共同事件，另有少數事件影響彼此重疊的股票群組。**潛在因子**就是其中一種共同變動；它不是任何人事先計算的欄位，而是從報酬共同變動的方式擷取出的模式。每檔股票都有一個**負載量**，表示它受該模式影響的程度。

此處擬合兩種擷取方式，兩者差異正是本節重點：

- [`13a_pca`](13a_pca.ipynb) 僅從報酬面板擷取因子。主成分分析尋找哪些股票組合能解釋最多共同變異，不需股票的其他資訊。負載量是附於個別股票的數值，在訓練窗口擬合後沿用。
- [`13b_ipca`](13b_ipca.ipynb) 則依股票的*特性*設定負載量。工具變數主成分分析使股票負載量成為可觀測特性的函數，所以特性相同的兩檔股票有相同負載方式，股票特性改變時負載量也會跟著改變。
<!-- ML4T_ZH_END c0000-p000 -->

**Why the second exists.** A loading attached to a stock says nothing about a stock that has not
been seen, and cannot move when the stock does. On a panel where names enter and leave and a
company's size and value change over a decade, that is a real limitation rather than a technical
one, and conditioning on characteristics is what removes it. What it costs is a stronger
assumption: that the relation between characteristics and loadings is stable, and is the same
for every stock.

**This notebook runs nothing.** It is the index over the two that do: it names them, opens what
they published, and shows that both are complete. Which of the two is worth more is a predictive
question, and it is answered in [`15_model_analysis`](15_model_analysis.ipynb).

**Learning objectives.** By the end of this notebook you will be able to:

- Say what a latent factor and a loading are, in terms of a panel of returns rather than of an
  algorithm.
- State the difference between a loading attached to a stock and a loading conditioned on the
  stock's characteristics, and name a situation in which only the second can answer.
- Say what the conditioned version assumes in exchange, and when that assumption would be
  uncomfortable.
- Read a table of published latent-factor results and tell a complete one from an incomplete one.

**Book reference**: Chapter 13.

**Prerequisites**: [`13a_pca`](13a_pca.ipynb) and [`13b_ipca`](13b_ipca.ipynb) have published the
results this index reads.

**What it writes**: nothing. It reads.

<!-- ML4T_ZH_BEGIN c0000-p001 -->
**為何需要第二種方法。** 附於某檔股票的負載量無法說明從未見過的股票，也無法隨股票本身變化。在股票進出面板、公司規模與價值歷經十年變動的資料中，這是實質限制，而非技術細節；以特性為條件便可解除此限制。代價是更強的假設：特性與負載量的關係穩定，且對每檔股票都相同。

**本 Notebook 不執行模型。** 它是另外兩份執行 Notebook 的索引：列出名稱、開啟已發布的結果，並確認兩者都已完成。哪一種更有價值是預測問題，由 [`15_model_analysis`](15_model_analysis.ipynb) 回答。

**學習目標。** 完成本 Notebook 後，你將能夠：

- 從報酬面板而非演算法的角度，說明潛在因子與負載量。
- 說明附於股票的負載量與依股票特性決定的負載量有何差異，並指出只有後者能處理的情境。
- 說明條件式方法為此接受的假設，以及何時該假設令人存疑。
- 閱讀已發布的潛在因子結果表，區分完整與未完成結果。

**書籍參考**：第 13 章。

**先決條件**：[`13a_pca`](13a_pca.ipynb) 與 [`13b_ipca`](13b_ipca.ipynb) 已發布本索引讀取的結果。

**產出內容**：無；只讀取資料。
<!-- ML4T_ZH_END c0000-p001 -->


In [1]:
"""Reference index for the latent-factor execution notebooks."""

import os
from pathlib import Path

import polars as pl

from case_studies.research import Study, open_study

In [2]:
CASE_STUDY_ID = "us_equities_panel"
EXECUTION_TIER = "canonical"
WORKSPACE = "experiments"

## What the two notebooks published

One row per label per factor model. Read it for two things.

**Both models present at every label.** A label carrying a PCA row and no IPCA row means the
second notebook did not finish there, and the comparison in
[`15_model_analysis`](15_model_analysis.ipynb) would then be measuring a difference between
labels rather than between factor models.

**`cv_identity` the same across the rows being compared.** It records which walk-forward design
a result was fitted and scored under. Two rows with different values measured themselves over
different windows, and ranking them is not a comparison.

Canonical execution reads the released study; preview execution reads an isolated workspace.

<!-- ML4T_ZH_BEGIN c0003-p000 -->
## 兩份 Notebook 發布的內容

每個標籤與因子模型組合各占一列。閱讀時要看兩件事。

**每個標籤都有兩種模型。** 若某標籤有 PCA 列卻沒有 IPCA 列，代表第二份 Notebook 未完成該標籤；此時 [`15_model_analysis`](15_model_analysis.ipynb) 的比較量測的是標籤差異，而非因子模型差異。

**比較的資料列具有相同的 `cv_identity`。** 它記錄結果擬合與評分所用的前進式驗證設計。兩列若值不同，就是在不同窗口量測，不能據此排序比較。

標準執行讀取已發布的研究；預覽執行讀取獨立工作區。
<!-- ML4T_ZH_END c0003-p000 -->


In [3]:
if EXECUTION_TIER == "canonical":
    # `Study.open` with no workspace, not `open_study`: this notebook writes nothing, and that is
    # the call that opens the released study read-only. `open_study` opens it for regeneration.
    study = Study.open(CASE_STUDY_ID)
elif EXECUTION_TIER == "preview":
    study = open_study(
        CASE_STUDY_ID,
        execution_tier=EXECUTION_TIER,
        workspace=Path(os.environ.get("ML4T_OUTPUT_DIR") or WORKSPACE),
    )
else:
    raise ValueError(f"Unsupported execution tier: {EXECUTION_TIER!r}")

latent_results = (
    study.predictions.table(include_preview=EXECUTION_TIER == "preview")
    .filter(
        (pl.col("family") == "latent_factors")
        & (pl.col("split") == "validation")
        & (pl.col("execution_tier") == EXECUTION_TIER)
        & pl.col("complete")
    )
    .select(
        "label",
        "config_name",
        "checkpoint_kind",
        "checkpoint_value",
        "cv_identity",
        "training_hash",
        "prediction_hash",
    )
    .sort("label", "config_name", "checkpoint_kind", "checkpoint_value")
)
# This notebook indexes what `13a_pca` and `13b_ipca` register and computes nothing of its
# own, so it has to run after them. Nothing else enforces that: the filter returns an empty
# frame rather than raising, the frame is the notebook's only result, and a render whose
# single result cell is blank is indistinguishable from a clean run. Empty is a different
# condition from the partial one the prose below anticipates - "if a row is missing above,
# run the notebook that produces it" expects some rows and got none, which means no
# latent-factor model has been fitted at all.
if latent_results.is_empty():
    raise ValueError(
        "no latent_factors predictions are registered for this execution tier: run "
        "13a_pca and 13b_ipca first. This notebook only indexes what they publish."
    )
latent_results

label,config_name,checkpoint_kind,checkpoint_value,cv_identity,training_hash,prediction_hash
str,str,str,i64,str,str,str
"""fwd_ret_1d""","""ipca""","""epoch""",0,"""78f37292f4c7ec32""","""765d7781e750""","""483a2d6f8228"""
"""fwd_ret_1d""","""pca""","""epoch""",0,"""78f37292f4c7ec32""","""a6217a86fc73""","""ab2c7bb27aa9"""


## What happens next

If a row is missing above, run the notebook that produces it - the execution notebooks reuse an
identity that already exists rather than refitting it, so re-running is cheap and safe.

[`15_model_analysis`](15_model_analysis.ipynb) reads these results alongside the other model
families and asks which ranks the cross-section better.
[`16_backtest`](16_backtest.ipynb) backtests every one of them. This index chooses nothing, and
the number of factors is not tuned anywhere in this case study: each model's preset declares one
and a sweep over that count would be a different experiment.

<!-- ML4T_ZH_BEGIN c0005-p000 -->
## 下一步

若上方缺少資料列，請執行產生該列的 Notebook；執行 Notebook 會重用既有識別資訊，不會重新擬合，因此重新執行成本低且安全。

[`15_model_analysis`](15_model_analysis.ipynb) 將這些結果與其他模型類別一併讀取，檢視哪種模型更能對橫斷面排序。[`16_backtest`](16_backtest.ipynb) 對每項結果進行回測。此索引不做選擇；案例研究也沒有調整因子數量：每個模型的預設設定都宣告一個數量，掃描不同數量會是另一項實驗。
<!-- ML4T_ZH_END c0005-p000 -->


## What to notice

**The two models answer the same question with different information.** PCA sees only how the
returns moved together; IPCA is additionally told what each stock is. Any difference between
them is what the characteristics were worth, on this panel, under the assumption that the
relation between characteristics and loadings holds across stocks and over time.

**A factor model is a compression, and a compression discards.** A handful of factors summarise
a three-thousand-name panel, so whatever is specific to one stock is by construction not in the
prediction. That is the trade being made rather than a defect: the models before this one are
where stock-specific information lives.

**Known limitations.** The factor count is declared, not searched, so nothing here says the
declared one is right - only what it gives. Both models are fitted on training windows only and
scored on validation folds that have been read many times over by the time a case study reaches
this notebook. And a latent factor has no name: it is a direction in the returns, and reading an
economic story into it is an interpretation this notebook does not support.

<!-- ML4T_ZH_BEGIN c0006-p000 -->
## 觀察重點

**兩種模型使用不同資訊回答相同問題。** PCA 只看到報酬如何共同變動；IPCA 另外取得各股票的特性。兩者差異表示，在特性與負載量的關係可跨股票及時間維持的假設下，特性在此面板上的價值。

**因子模型是一種壓縮，壓縮必有取捨。** 少數因子概括三千檔股票的面板，因此單一股票特有的資訊，依建構方式就不在預測之中。這是取捨而非缺陷；前面的模型才處理個股資訊。

**已知限制。** 因子數量是事先宣告而非搜尋得來，因此此處只能呈現該數量的結果，無法證明它正確。兩種模型都只在訓練窗口擬合，並在案例研究進展至此時已被反覆查看的驗證折評分。此外，潛在因子沒有名稱：它是報酬空間中的一個方向；替它賦予經濟故事，超出本 Notebook 可支持的詮釋。
<!-- ML4T_ZH_END c0006-p000 -->
